In [2]:
!pip install -U langchain langchain-core langchain-community langchain-openai langchain-text-splitters langsmith langchainhub chromadb langgraph

In [3]:
!pip install -U transformers accelerate sentencepiece

In [4]:
from transformers.pipelines import SUPPORTED_TASKS
print(SUPPORTED_TASKS.keys())

dict_keys(['audio-classification', 'automatic-speech-recognition', 'text-to-audio', 'feature-extraction', 'text-classification', 'token-classification', 'table-question-answering', 'document-question-answering', 'fill-mask', 'text-generation', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-audio-classification', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'object-detection', 'zero-shot-object-detection', 'depth-estimation', 'video-classification', 'mask-generation', 'keypoint-matching', 'any-to-any'])


In [5]:
from google.colab import userdata
import os
os.environ['OPENAI_API_KEY'] = userdata.get('mytestopenAI')


In [6]:
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(web_path = ["https://www.livelaw.in/top-stories/supreme-court-dismisses-congress-meenakshi-natarajans-plea-against-rajya-sabha-nomination-rejection-537631"])
docs = loader.load()
print(docs)

/tmp/ipykernel_24481/3020609101.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


[Document(metadata={'source': 'https://www.livelaw.in/top-stories/supreme-court-dismisses-congress-meenakshi-natarajans-plea-against-rajya-sabha-nomination-rejection-537631', 'title': "Supreme Court Dismisses Meenakshi Natarajan's Plea Against Rajya Sabha Nomination Rejection", 'description': 'The Supreme Court on Friday (June 12) dismissed the writ petition filed by Congress member Meenakshi Natarajan challenging the rejection of her Rajya Sabha candidature from Madhya Pradesh, and gave...', 'language': 'en'}, page_content='Supreme Court Dismisses Meenakshi Natarajan\'s Plea Against Rajya Sabha Nomination Rejection                               LoginAccountSubscribe PremiumMoreTop StoriesSupreme CourtSearchLiveLaw HindiTop StoriesSupreme CourtSupreme Court  SC JudgementsHigh CourtHigh Court  All High CourtsAllahabad High CourtAndhra Pradesh High CourtBombay High CourtCalcutta High CourtChhattisgarh High CourtDelhi High CourtGauhati High CourtGujarat High CourtHimachal Pradesh High Cou

In [7]:
!pip install langchain-text-splitters


In [8]:
!pip install -q langchain-text-splitters

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100)
splits = text_splitter.split_documents(docs)

In [10]:
print(splits[3])
print(splits[4])


page_content='Supreme Court Dismisses Meenakshi Natarajan's Plea Against Rajya Sabha Nomination Rejection, Allows Her To File Election PetitionGursimran Kaur Bakshi 12 Jun 2026 1:26 PM ISTShare this      Listen to this Article The Supreme Court on Friday (June 12) dismissed the writ petition filed by Congress member Meenakshi Natarajan challenging the rejection of her Rajya Sabha candidature from Madhya Pradesh, and gave her liberty to raise the challenge in an election petition filed in terms of the Representation of the People Act.A bench comprising Justice Prashant Kumar Mishra and Justice AS Chandurkar declined the exercise of its writ jurisdiction citing the Constitutional bar as per Article 329. The writ petition was accordingly dismissed as non-maintainable.The bench rejected the argument of the petitioner that Article 32 can be invoked to cure "glaring and manifest" errors in the rejection of nomination. "If the Court accepts such arguments to find out glaring cases which are r

In [11]:
print(len(splits))

15


In [16]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())

In [ ]:
print(vectorstore._collection.count())

In [ ]:
print(vectorstore._collection.get())

In [18]:
print("\nCollection 1 - ",vectorstore._collection.get(ids=['2a825ac4-f413-44d2-a629-3c29dd64e864'],include =['embeddings','documents']))


**RAG** **Pipeline**

In [ ]:
retriever = vectorstore.as_retriever()

In [ ]:
from langchain import hub
prompt = hub.pull("rlm/rag-prompt")

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI()

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
def format_docs(docs):
  return '\n'.join(doc.page_content for doc in docs)



In [ ]:
rag_chain  = ({'context': retriever | format_docs, 'question':RunnablePassthrough()}
             |prompt
             |llm
             |StrOutputParser()
             )

In [ ]:
rag_chain.invoke("supreme court order")

In [ ]:
from langchain_core.runnables import RunnablePassthrough

def print_prompt(prompt_text):
  print("Prompt - ",prompt_text)
  return prompt_text

In [ ]:
rag_chain_with_print  = ({'context': retriever | format_docs, 'question':RunnablePassthrough()}
             |prompt
             |RunnablePassthrough(print_prompt)
             |llm
             |StrOutputParser()
             )

In [ ]:
rag_chain_with_print.invoke("supreme court order")